# Bootstrap

This bootstrap will guide you through the 4 main steps of the project:
- Data gathering
- Database organization
- Big Data analysis
- Interactive visualizations

> All datasets needed for this bootstrap are available on [the sharepoint site T-DAT-902](https://epitechfr.sharepoint.com/sites/TDAT902/Documents%20partages/Forms/AllItems.aspx)


## Non tabular data gathering

Beside collection of basic csv files, you might also need to do some scraping to extract your data (from
websites like pap.fr or leboncoin.fr).

As a useful start, you can try to extract and structure data located on ville-ideale.fr, a website where people
can review the city they live(d) in. Try to extract data from as many cities as possible.

Create a dataset to store everything you collected using tools adapted for non tabular data.

> Start with using some official metadata to get information about the cities, their name and link them
to the scrapped data

The following website provides a nice tutorial on how to use the "Beautiful Soup" library for scraping with Python : [laconsole.dev](https://laconsole.dev/formations/python/scraping-beautiful-soup)

### What is Web Scraping ?

Data scraping is an essential technique for data extraction on websites using Python.

We distinguish two main types of scraping on the web:

1. The SERP scraping (Search Engine Result Page) : aims to collect informations on the search results, as the titles, descriptions, URLs and other metadatas. This type of scraping is widely exploited by SSEO tools like SEMrush or Ahrefs which analyses the keywords performance, tacks the rankings and monitors concurrents in the search engines.

2. Website scraping : aims to collect specific data on one or many web pages, such as products prices, customer reviews, blog posts or any publicly accessible data.

No matter the type of scraping, it has numerous use cases such as collecting data, curating content, feeding machine learning, etc...

We invite you to read our [web scraping guide](https://laconsole.dev/blog/guide-web-scraping) if you want to learn more about this fascinating domain.

## Get the content of a web page with `requests`

As you may have understood the wbe scraping consists in analysing web pages content. We need to be able to retrieve web pages content from their URL! And this is the role of our first mandatory Python module: `requests`

In [2]:
!uv add requests

Resolved 39 packages in 1ms
Audited 34 packages in 0.04ms


In [3]:
response = None

In [11]:
import requests

if response is None:
    print("Sending request to ville-ideale.fr...")
    response = requests.get("https://www.ville-ideale.fr/creteil_94028?page=3#commentaires")
    print(f"Response text: '{response.text}'")
else:
    print(f"Response text: \n'{response.text[:333]}'")


Response text: 
'<!DOCTYPE html>
<html lang="fr"><head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
'meta name="description" content="CRETEIL : notes et avis sur cette ville. Environnement, transports, santé, sécurité, sports et loisirs, culture, enseignement, commerces, qualité de vie." />


In [12]:
if response.status_code == 200:
    print(response.status_code)
else:
    print("Error during request:", response.status_code)

200


### Analysing a webpage's content with `beautifulsoup`

Now let's analyze the returned HTML code with [BeautifulSoup](https://beautiful-soup-4.readthedocs.io/en/latest/)

In [14]:
!uv add beautifulsoup4

Resolved 42 packages in 168ms                                        
Installed 3 packages in 4ms.3                                    
 + beautifulsoup4==4.14.3
 + soupsieve==2.8.3
 + typing-extensions==4.15.0


#### Stirring the soup !

The "soup" is simply the name given to the object that is going to store the DOM structure of the page we are scraping.

In [17]:
from bs4 import BeautifulSoup

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    formatted_html = soup.prettify()
else:
    print("Error during request:", response.status_code)

print(formatted_html[:500])

<!DOCTYPE html>
<html lang="fr">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="CRETEIL : notes et avis sur cette ville. Environnement, transports, santé, sécurité, sports et loisirs, culture, enseignement, commerces, qualité de vie." name="description"/>
  <title>
   Avis sur CRETEIL : la ville idéale ?
  </title>
  <script src="/scripts/ville.js">
  </script>
  <link href="/styles/ville.css" media="screen" rel="style


The `BeautifulSoup` receives 2 arguments:

- The raw HTML code (contained in `response.text`) that we want to scrape
- A parser for analyzing the HTML content, by default we'll use `html.parser`

BeautifulSoup has 5 libraries for HTML code analysis : `html.parser`, `xml`, `lxml`, `lxml-xml`and `html5lib`.

The choice of a parser over another depends on the type of code to parse (XML or HTML), the performances or even compatibility reasons.

> XML (eXtensible Markup Language) is a markup language used to store and transport data in a flexible and structured way. HTML is a specific use case of HTML to structure web pages content.

Once we've created our `BeautifulSoup` object (generally named `soup`), it's possible to explore the DOM tree (Document Object Model) of the HTML page.

#### Making our soup more digest

The `prettify()` method can be used to format the HTML code in a readable way, by adding indentations and line returns.

> **Warning**: This method returns a string of formatted HTML, this is not a usable `BeautifulSoup` object for scraping

### Exploring the soup

#### An object soup

BeautifulSoup allows us to navigate easily the HTML structure, represented as an object soup.

Those objects allows us to retrieve, analyze and manipulate the HTML content in a very flexible way.

Here's the 3 main objects that we'll encounter:
- `Tag`: represents an HTML tag, like `<div>`, `<p>` or `<a>`. It allows us to access the content of a tag, it's childs or it's attributes
- `NavigableString`: represents the text content inside an HTML tag
- `Comment`: represents an HTML comment

The `Tag` represents the vast majority of the manipulated objects



In [ ]:
city = soup.section.h2.contents[0].split(' ')[4]
number_of_pages = soup.section.h4.contents[0].split('/')[1].strip()

print(f"The number of pages about {city} is: {number_of_pages}")

The number of pages about Créteil is: 34


In [123]:
from dataclasses import dataclass, asdict

@dataclass
class Critere:
    categorie: str
    note: int

@dataclass
class Note:
    moyenne: float
    detail: list[Critere]

@dataclass
class Commentaires:
    positif: str
    negatif: str

@dataclass
class Avis:
    date: str
    username: str
    note: Note
    commentaires: Commentaires


In [126]:
comms = list(soup.section.find_all(attrs={"class": "comm"}))

avis: list[Avis] = []

for comm in comms:
    date: str = comm.p.span.contents[0]
    username: str = comm.p.strong.contents[0]

    moyenne: float = comm.find('strong', attrs={'class': 'moyenne'}).contents[0]
    detail: list[Critere] = []

    libelles = ["Environnement", "Transports", "Sécurité", "Sport et loisirs", "Culture", "Enseignement", "Commerces", "Qualité de vie"]
    notes = map(lambda n: n.contents[0], comm.table.tr.find_next_sibling('tr'))

    for critere in zip(libelles, notes):
        detail.append(Critere(categorie = critere[0], note = critere[1]))

    note = Note(moyenne, detail)

    points = list(comm.p.find_next_siblings('p'))
    point_positifs = ' '.join([p.text for p in points[0].contents[1:]])
    point_negatifs = ' '.join([p.text for p in points[1].contents[1:]])

    commentaires = Commentaires(point_positifs, point_negatifs)

    avis.append(Avis(date, username, note, commentaires))

import json

for a in avis:
    b = asdict(a)
    print(json.dumps(b, indent=True))


{
 "date": "Avis post\u00e9 le 19-03-2025 \u00e0 18:32",
 "username": "Juliette94",
 "note": {
  "moyenne": "8.69",
  "detail": [
   {
    "categorie": "Environnement",
    "note": "7"
   },
   {
    "categorie": "Transports",
    "note": "10"
   },
   {
    "categorie": "S\u00e9curit\u00e9",
    "note": "7"
   },
   {
    "categorie": "Sport et loisirs",
    "note": "8"
   },
   {
    "categorie": "Culture",
    "note": "10"
   },
   {
    "categorie": "Enseignement",
    "note": "7"
   },
   {
    "categorie": "Commerces",
    "note": "8"
   },
   {
    "categorie": "Qualit\u00e9 de vie",
    "note": "10"
   }
  ]
 },
 "commentaires": {
  "positif": "Nous avons emm\u00e9nag\u00e9 \u00e0 Cr\u00e9teil il y a six ans et n'avons jamais regrett\u00e9 ce choix. Mon avis ne concerne que les quartiers du secteur sud  \r\n(la source,  pointe du lac,  le Port,  Front du lac),  je sais que c'est loin de d\u00e9crire toute la ville. Mais,  en ce qui nous concerne,  c'est un sans fautes.  \r\nNou